<a href="https://colab.research.google.com/github/FatherNurt/FUNt-Cosmologiical-Model-of-All-Things/blob/main/FUNt_Diagnostic_Plug_In_Toy_Model_v0_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# FUNt Diagnostic Plug-In Toy Model v0.3
# Five-operator recurrence diagnostic
# Full corpus: https://zenodo.org/search?q=metadata.creators.person_or_org.name%3A%22Nowlin%2C%20Michael%20K.%22
# ============================================================

def validate_R(R):
    if len(R) < 2:
        return False, "Need at least 2 points for gradient."
    if not all(r > 0 for r in R):
        return False, "R contains non-positive value(s)."
    if not all(R[i] < R[i+1] for i in range(len(R)-1)):
        return False, "R is not strictly increasing."
    return True, "R validated: positive, strictly increasing, ratio-scale."

def run_funt_diagnostic(R, S):
    print("=" * 66)
    print("FUNt DIAGNOSTIC REPORT — Toy Model v0.3")
    print("Full corpus: https://zenodo.org/search?q=metadata.creators.person_or_org.name%3A%22Nowlin%2C%20Michael%20K.%22")
    print("=" * 66)
    print("Operators used: H=0, Log–Ψ, 2√, h³π, hHRT (strictly only these)")
    print("Scope: Internal recurrence consistency diagnostic — conservative scoping")
    print("-" * 66)

    # === R VALIDATION ===
    valid, msg = validate_R(R)
    print("\n[R VALIDATION]")
    print(msg)
    if not valid:
        print("Analysis halted — please correct R sequence.")
        return

    n = len(R)
    print(f"Points supplied: {n}")

    # === H = 0 ANCHOR ===
    print("\n[H = 0 ANCHOR]")
    print("Note: This run uses the first S value as a provisional reference.")
    print("For real data, define an explicit hydrogen ground-state anchor.")

    # === COMPUTE Log–Ψ ===
    dR = [R[i+1] - R[i] for i in range(n-1)]
    dS = [S[i+1] - S[i] for i in range(n-1)]
    Psi = []
    for i in range(n-1):
        if dR[i] != 0:
            Psi.append(dS[i] / dR[i])
        else:
            Psi.append(0.0)

    print("\n[Log–Ψ RECURRENCE GRADIENT]")
    print("Ψ = dS / dR (finite difference)")
    print("Interval |   R_start → R_end   |    Ψ     | Tendency")
    print("-" * 58)
    for i in range(n-1):
        if Psi[i] > 0:
            tend = "growth/expansion (+)"
        elif Psi[i] < 0:
            tend = "decay/relaxation (-)"
        else:
            tend = "quasi-stationary (~0)"
        print(f"  {i+1:2d}    | {R[i]:7.3f} → {R[i+1]:7.3f} | {Psi[i]:+8.4f} | {tend}")

    n_growth = sum(1 for p in Psi if p > 0)
    n_relax  = sum(1 for p in Psi if p < 0)
    n_stat   = sum(1 for p in Psi if abs(p) < 1e-8)
    print(f"\nSummary: {n_growth} growth | {n_relax} relaxation | {n_stat} quasi-stationary intervals")

    # === 2√ CONSTRAINED TRANSPORT ===
    print("\n[2√ CONSTRAINED TRANSPORT]")
    if n > 3:
        avg_abs_dS = sum(abs(x) for x in dS) / len(dS)
        max_abs_dS = max(abs(x) for x in dS)
        ratio = max_abs_dS / avg_abs_dS if avg_abs_dS > 0 else 0
        if ratio < 2.2:
            transport_note = "Rate of change remains within a coherent envelope. No abrupt jumps detected."
        elif ratio < 3.5:
            transport_note = "Moderate variation in rate observed; still within plausible regime limits for toy data."
        else:
            transport_note = "Elevated variation in rate detected. Transport coherence should be examined more closely."
    else:
        transport_note = "Too few points for robust transport assessment."
    print(transport_note)

    # === h³π REGIME TRANSITION SCAN (conservative) ===
    print("\n[h³π REGIME TRANSITION SCAN]")
    sign_changes = []
    for i in range(1, len(Psi)):
        if (Psi[i-1] > 0 and Psi[i] < 0) or (Psi[i-1] < 0 and Psi[i] > 0):
            sign_changes.append(i)
    if sign_changes:
        last_idx = sign_changes[-1]
        last_R = (R[last_idx] + R[last_idx + 1]) / 2
        last_Psi = Psi[last_idx]
        if abs(last_Psi) > 0.06 and (n - 1 - last_idx) >= 2:
            trans_note = (f"Potential relaxation onset near R ≈ {last_R:.3f} "
                          f"(Ψ sign change at interval {last_idx+1}).\n"
                          "Conservative reading: Sign reversal present and sustained. "
                          "However, magnitude is modest and no clear transport/coherence failure "
                          "or stabilizing re-indexing is observed. Additional points recommended "
                          "before asserting a genuine h³π regime transition.")
        else:
            trans_note = ("Isolated or weak sign change noted. Insufficient evidence for "
                          "h³π transition consideration under conservative criteria.")
    else:
        trans_note = "No Ψ sign change detected. No h³π candidate identified."
    print(trans_note)

    # === hHRT RELAXATION / RETURN TIME ===
    print("\n[hHRT RELAXATION / RETURN TIME]")
    if n_relax > 0:
        relax_streak = 0
        for p in reversed(Psi):
            if p < 0:
                relax_streak += 1
            else:
                break
        if relax_streak >= 2:
            hrt_note = (f"Relaxation phase active for last {relax_streak} interval(s). "
                        "No return toward baseline or sign reversal observed within current window. "
                        "hHRT not yet resolved — relaxation appears ongoing.")
        else:
            hrt_note = "Brief relaxation interval(s) present. Return behavior not yet observable."
    else:
        hrt_note = "No relaxation phase detected in this window."
    print(hrt_note)

    # === OVERALL REGIME CHARACTER ===
    print("\n[OVERALL REGIME CHARACTER — CONSERVATIVE SUMMARY]")
    if n_growth > n_relax and n_relax == 0:
        regime = "Dominant growth/expansion tendency across the window."
    elif n_relax > n_growth and n_growth == 0:
        regime = "Dominant relaxation/decay tendency across the window."
    elif n_relax > 0 and n_growth > 0:
        regime = ("Mixed regime: growth phase followed by relaxation onset. "
                  "Early transition signal present but insufficient evidence for confirmed "
                  "h³π boundary under strict criteria. More data recommended.")
    else:
        regime = "Quasi-stationary or mixed with no dominant tendency resolved."
    print(regime)

    print("\n" + "=" * 66)
    print("END OF REPORT — Analysis used strictly the five defined operators.")
    print("This is an internal consistency diagnostic for recurrence data only.")
    print("=" * 66)

# ============================================================
# === USER DATA: PLUG YOUR TOY MODEL HERE ====================
# ============================================================
R = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]          # <-- plug your R here
S = [1.00, 1.40, 1.95, 2.55, 2.80, 2.75, 2.60, 2.45]  # <-- plug your S here

run_funt_diagnostic(R, S)

FUNt DIAGNOSTIC REPORT — Toy Model v0.3
Full corpus: https://zenodo.org/search?q=metadata.creators.person_or_org.name%3A%22Nowlin%2C%20Michael%20K.%22
Operators used: H=0, Log–Ψ, 2√, h³π, hHRT (strictly only these)
Scope: Internal recurrence consistency diagnostic — conservative scoping
------------------------------------------------------------------

[R VALIDATION]
R validated: positive, strictly increasing, ratio-scale.
Points supplied: 8

[H = 0 ANCHOR]
Note: This run uses the first S value as a provisional reference.
For real data, define an explicit hydrogen ground-state anchor.

[Log–Ψ RECURRENCE GRADIENT]
Ψ = dS / dR (finite difference)
Interval |   R_start → R_end   |    Ψ     | Tendency
----------------------------------------------------------
   1    |   1.000 →   2.000 |  +0.4000 | growth/expansion (+)
   2    |   2.000 →   3.000 |  +0.5500 | growth/expansion (+)
   3    |   3.000 →   4.000 |  +0.6000 | growth/expansion (+)
   4    |   4.000 →   5.000 |  +0.2500 | growth/